# 01_target_definition_v2.ipynb
## 변경 사항 (v1 → v2)
- 기존 피처는 100% 유지
- 구매 트렌드, 다양성 변화, 구매 리듬 피처 7개 추가
- 모든 추가 피처는 cut_off 이전 데이터만 사용 (누수 없음)

## 목차
1. 데이터 로드
2. 고객별 최대 구매 간격 계산
3. Cut-off 기준일 확정
4. 라벨(Churn) 생성
5. 파생 피처 생성
   - 5-1. 기본 RFM
   - 5-2. Recency
   - 5-3. 활동성 (90일 / 180일)
   - 5-4. 구매 간격
   - 5-5. 구매 다양성
   - 5-6. 취향
   - 5-7. [NEW] 구매 트렌드
   - 5-8. [NEW] 다양성 변화
   - 5-9. [NEW] 구매 리듬
6. 데이터 결합 (Merge)
7. 데이터 무결성 / 누수 검증
8. 데이터 저장
9. 최종 데이터셋 정의서


## 1. 데이터 로드

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rc('font', family='Malgun Gothic')  # Mac: 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid')

trans     = pd.read_csv('./raw/transactions_train.csv', nrows=100000)
customers = pd.read_csv('./raw/customers.csv')
articles  = pd.read_csv('./raw/articles.csv')

print(f'Transactions : {trans.shape}')
print(f'Customers    : {customers.shape}')
print(f'Articles     : {articles.shape}')


Transactions : (100000, 5)
Customers    : (1371980, 7)
Articles     : (105542, 25)


## 2. 고객별 최대 구매 간격 계산

In [3]:
trans_all = pd.read_csv('./raw/transactions_train.csv', usecols=['t_dat', 'customer_id'])
trans_all['t_dat'] = pd.to_datetime(trans_all['t_dat'])

trans_sorted = trans_all.sort_values(['customer_id', 't_dat'])
trans_sorted['diff'] = trans_sorted.groupby('customer_id')['t_dat'].diff().dt.days
user_max_diff = trans_sorted.groupby('customer_id')['diff'].max().dropna()
print(f'구매 간격 계산 완료: {len(user_max_diff):,}명')


구매 간격 계산 완료: 1,230,767명


## 3. Cut-off 기준일 확정

In [4]:
median_diff    = user_max_diff.median()
mean_diff      = user_max_diff.mean()
q75_days_calc  = user_max_diff.quantile(0.75)

print(f'중앙값  : {median_diff}일')
print(f'평균값  : {mean_diff:.1f}일')
print(f'75분위수: {q75_days_calc}일')

q75_days     = int(q75_days_calc)
min_date     = trans_all['t_dat'].min()
max_date     = trans_all['t_dat'].max()
cut_off_date = max_date - pd.Timedelta(days=q75_days)

print(f'\ncut_off_date : {cut_off_date.date()}')
print(f'관측 기간    : {q75_days}일')


중앙값  : 112.0일
평균값  : 131.9일
75분위수: 203.0일

cut_off_date : 2020-03-03
관측 기간    : 203일


## 4. 라벨(Churn) 생성

In [5]:
df_past_base   = trans_all[trans_all['t_dat'] <= cut_off_date]
df_future_base = trans_all[trans_all['t_dat'] >  cut_off_date]

user_stats = df_past_base.groupby('customer_id')['t_dat'].agg(['min','max']).reset_index()
user_stats.columns = ['customer_id','first_buy','last_buy_past']
user_stats['tenure'] = (cut_off_date - user_stats['first_buy']).dt.days

df_label = user_stats[user_stats['tenure'] >= 30].copy()

active_users_in_future = df_future_base['customer_id'].unique()
df_label['churn'] = df_label['customer_id'].isin(active_users_in_future).apply(lambda x: 0 if x else 1)
df_label = df_label[['customer_id','churn']]

print(f'분석 대상 고객 수: {len(df_label):,}명')
print(df_label['churn'].value_counts())
print(f'이탈률: {df_label["churn"].mean()*100:.2f}%')


분석 대상 고객 수: 1,143,652명
churn
1    572053
0    571599
Name: count, dtype: int64
이탈률: 50.02%


## 5. 파생 피처 생성

모든 피처는 cut_off 이전 데이터만 사용합니다.

In [6]:
trans_feat = pd.read_csv('./raw/transactions_train.csv',
                         usecols=['t_dat','customer_id','article_id','price'])
trans_feat['t_dat'] = pd.to_datetime(trans_feat['t_dat'])
trans_feat = trans_feat[trans_feat['t_dat'] <= cut_off_date].copy()
print(f'피처 생성 대상 거래 수: {len(trans_feat):,}건')


피처 생성 대상 거래 수: 22,991,230건


### 5-1. 기본 RFM 피처

In [7]:
user_rfm = trans_feat.groupby('customer_id').agg(
    frequency  = ('article_id','count'),
    price_mean = ('price','mean'),
    price_sum  = ('price','sum'),
    price_max  = ('price','max'),
).reset_index()
print(f'RFM 피처 생성 완료: {user_rfm.shape}')


RFM 피처 생성 완료: (1169862, 5)


### 5-2. Recency 피처

In [8]:
user_last_buy = trans_feat.groupby('customer_id')['t_dat'].max().reset_index()
user_last_buy.columns = ['customer_id','last_buy_before_cutoff']
user_last_buy['recency'] = (cut_off_date - user_last_buy['last_buy_before_cutoff']).dt.days
user_last_buy = user_last_buy.drop(columns=['last_buy_before_cutoff'])
print(f'Recency 피처 생성 완료: {user_last_buy.shape}')


Recency 피처 생성 완료: (1169862, 2)


### 5-3. 활동성 피처 (90일 / 180일)

In [9]:
date_90d  = cut_off_date - pd.Timedelta(days=90)
date_180d = cut_off_date - pd.Timedelta(days=180)

freq_90d = (
    trans_feat[trans_feat['t_dat'] >= date_90d]
    .groupby('customer_id')['article_id'].count()
    .reset_index().rename(columns={'article_id':'frequency_last_90d'})
)
freq_180d = (
    trans_feat[trans_feat['t_dat'] >= date_180d]
    .groupby('customer_id')['article_id'].count()
    .reset_index().rename(columns={'article_id':'frequency_last_180d'})
)
print(f'90일  활동 고객: {len(freq_90d):,}명')
print(f'180일 활동 고객: {len(freq_180d):,}명')


90일  활동 고객: 485,365명
180일 활동 고객: 704,745명


### 5-4. 구매 간격 피처

In [10]:
trans_sorted_feat = trans_feat.sort_values(['customer_id','t_dat'])
trans_sorted_feat['interval'] = (
    trans_sorted_feat.groupby('customer_id')['t_dat'].diff().dt.days
)
user_interval = trans_sorted_feat.groupby('customer_id')['interval'].agg(
    avg_purchase_interval = 'mean',
    std_purchase_interval = 'std',
).reset_index()
print(f'구매 간격 피처 생성 완료: {user_interval.shape}')


구매 간격 피처 생성 완료: (1169862, 3)


### 5-5. 구매 다양성 피처

In [11]:
trans_with_cat = pd.merge(
    trans_feat[['customer_id','article_id']],
    articles[['article_id','product_group_name']],
    on='article_id', how='left'
)
user_diversity = trans_with_cat.groupby('customer_id').agg(
    unique_articles       = ('article_id','nunique'),
    unique_product_groups = ('product_group_name','nunique'),
).reset_index()
print(f'다양성 피처 생성 완료: {user_diversity.shape}')


다양성 피처 생성 완료: (1169862, 3)


### 5-6. 취향 피처

In [12]:
trans_style = pd.merge(
    trans_feat[['customer_id','article_id']],
    articles[['article_id','index_group_name','product_group_name']],
    on='article_id', how='left'
)
def get_most_frequent(x):
    if x.empty: return 'NONE'
    return x.value_counts().idxmax()

user_style = trans_style.groupby('customer_id').agg(
    index_group_name   = ('index_group_name',   get_most_frequent),
    product_group_name = ('product_group_name', get_most_frequent),
).reset_index()
print(f'취향 피처 생성 완료: {user_style.shape}')


취향 피처 생성 완료: (1169862, 3)


### 5-7. [NEW] 구매 트렌드 피처

구매 빈도와 금액이 시간이 갈수록 줄어드는지를 측정합니다.
모두 cut_off 이전 데이터만 사용하므로 누수 없음.

- `freq_decay_rate` : 최근 90일 구매 / 최근 180일 구매 → 0.5면 후반에 구매가 절반으로 줄었다는 뜻
- `recent_price_ratio` : 최근 90일 평균 구매액 / 전체 평균 구매액 → 1보다 작으면 지출이 줄어드는 중


In [13]:
# 최근 90일 평균 구매액
price_90d = (
    trans_feat[trans_feat['t_dat'] >= date_90d]
    .groupby('customer_id')['price'].mean()
    .reset_index().rename(columns={'price':'price_mean_last_90d'})
)

# 구매 트렌드 피처 결합
user_trend = freq_90d.copy()
user_trend = pd.merge(user_trend, freq_180d,  on='customer_id', how='outer')
user_trend = pd.merge(user_trend, price_90d,  on='customer_id', how='left')
user_trend = pd.merge(user_trend, user_rfm[['customer_id','price_mean']], on='customer_id', how='left')

# 결측치 0으로 채우기 (해당 기간 구매 없는 고객)
user_trend['frequency_last_90d']  = user_trend['frequency_last_90d'].fillna(0)
user_trend['frequency_last_180d'] = user_trend['frequency_last_180d'].fillna(0)

# freq_decay_rate: 90일 구매 / 180일 구매
# 180일 구매가 0인 고객은 NaN 처리 (구매 이력 없음)
user_trend['freq_decay_rate'] = np.where(
    user_trend['frequency_last_180d'] > 0,
    user_trend['frequency_last_90d'] / user_trend['frequency_last_180d'],
    np.nan
)

# recent_price_ratio: 최근 90일 평균 구매액 / 전체 평균 구매액
user_trend['recent_price_ratio'] = np.where(
    user_trend['price_mean'] > 0,
    user_trend['price_mean_last_90d'] / user_trend['price_mean'],
    np.nan
)

user_trend = user_trend[['customer_id','freq_decay_rate','recent_price_ratio']]
print(f'구매 트렌드 피처 생성 완료: {user_trend.shape}')
print(user_trend.describe())


구매 트렌드 피처 생성 완료: (704745, 3)
       freq_decay_rate  recent_price_ratio
count    704745.000000       485365.000000
mean          0.471803            1.022851
std           0.405461            0.375835
min           0.000000            0.003731
25%           0.000000            0.831630
50%           0.450000            1.000000
75%           1.000000            1.142794
max           1.000000           10.268999


### 5-8. [NEW] 다양성 변화 피처

- `unique_articles_last_90d` : 최근 90일 구매 고유 상품 수
- `diversity_decay_rate` : 최근 90일 고유 상품 수 / 전체 고유 상품 수 → 낮을수록 쇼핑 스펙트럼이 줄어드는 중


In [14]:
# 최근 90일 고유 상품 수
diversity_90d = (
    trans_feat[trans_feat['t_dat'] >= date_90d]
    .groupby('customer_id')['article_id'].nunique()
    .reset_index().rename(columns={'article_id':'unique_articles_last_90d'})
)

user_diversity_change = pd.merge(diversity_90d, user_diversity[['customer_id','unique_articles']],
                                  on='customer_id', how='right')
user_diversity_change['unique_articles_last_90d'] = user_diversity_change['unique_articles_last_90d'].fillna(0)

# diversity_decay_rate
user_diversity_change['diversity_decay_rate'] = np.where(
    user_diversity_change['unique_articles'] > 0,
    user_diversity_change['unique_articles_last_90d'] / user_diversity_change['unique_articles'],
    np.nan
)

user_diversity_change = user_diversity_change[['customer_id','unique_articles_last_90d','diversity_decay_rate']]
print(f'다양성 변화 피처 생성 완료: {user_diversity_change.shape}')
print(user_diversity_change.describe())


다양성 변화 피처 생성 완료: (1169862, 3)
       unique_articles_last_90d  diversity_decay_rate
count              1.169862e+06          1.169862e+06
mean               2.346789e+00          1.528378e-01
std                4.869655e+00          2.773541e-01
min                0.000000e+00          0.000000e+00
25%                0.000000e+00          0.000000e+00
50%                0.000000e+00          0.000000e+00
75%                3.000000e+00          1.818182e-01
max                1.890000e+02          1.000000e+00


### 5-9. [NEW] 구매 리듬 피처

- `last_interval` : 마지막 구매 간격 (가장 최근 공백일 수)
- `interval_acceleration` : 마지막 구매 간격 / 평균 구매 간격 → 1보다 크면 최근 들어 구매가 뜸해진 것

구매가 1회뿐인 고객은 간격을 계산할 수 없으므로 NaN 유지 (트리 모델 위임).


In [15]:
# 고객별 마지막 구매 간격 추출
last_interval_df = (
    trans_sorted_feat.groupby('customer_id')['interval']
    .last()  # 시간순 정렬된 상태에서 마지막 간격
    .reset_index().rename(columns={'interval':'last_interval'})
)

user_rhythm = pd.merge(last_interval_df,
                        user_interval[['customer_id','avg_purchase_interval']],
                        on='customer_id', how='left')

# interval_acceleration: 마지막 간격 / 평균 간격
user_rhythm['interval_acceleration'] = np.where(
    user_rhythm['avg_purchase_interval'] > 0,
    user_rhythm['last_interval'] / user_rhythm['avg_purchase_interval'],
    np.nan
)

user_rhythm = user_rhythm[['customer_id','last_interval','interval_acceleration']]
print(f'구매 리듬 피처 생성 완료: {user_rhythm.shape}')
print(user_rhythm.describe())


구매 리듬 피처 생성 완료: (1169862, 3)
       last_interval  interval_acceleration
count   1.050786e+06          773546.000000
mean    1.659088e+01               0.949168
std     5.133018e+01               2.430658
min     0.000000e+00               0.000000
25%     0.000000e+00               0.000000
50%     0.000000e+00               0.000000
75%     0.000000e+00               0.824521
max     5.300000e+02             175.000000


## 6. 데이터 결합 (Merge)

In [16]:
customers_features = customers[[
    'customer_id','FN','Active',
    'club_member_status','fashion_news_frequency','age'
]]

df_analysis = df_label.copy()

for feat_df, name in [
    (user_rfm,              'RFM'),
    (user_last_buy,         'Recency'),
    (freq_90d,              '활동성 90일'),
    (freq_180d,             '활동성 180일'),
    (user_interval,         '구매 간격'),
    (user_diversity,        '다양성'),
    (user_style,            '취향'),
    (user_trend,            '[NEW] 구매 트렌드'),
    (user_diversity_change, '[NEW] 다양성 변화'),
    (user_rhythm,           '[NEW] 구매 리듬'),
    (customers_features,    '고객 메타'),
]:
    df_analysis = pd.merge(df_analysis, feat_df, on='customer_id', how='left')
    print(f'{name} 병합 완료 → shape: {df_analysis.shape}')

# 활동성 결측치 → 0 (해당 기간 구매 없음)
df_analysis['frequency_last_90d']  = df_analysis['frequency_last_90d'].fillna(0).astype(int)
df_analysis['frequency_last_180d'] = df_analysis['frequency_last_180d'].fillna(0).astype(int)
df_analysis['unique_articles_last_90d'] = df_analysis['unique_articles_last_90d'].fillna(0).astype(int)

# 메모리 정리
del trans_feat, trans_sorted_feat, user_rfm, user_last_buy
del freq_90d, freq_180d, user_interval, user_diversity, user_style
del user_trend, user_diversity_change, user_rhythm

print(f'\n최종 데이터 크기: {df_analysis.shape}')
print(f'컬럼 목록: {list(df_analysis.columns)}')


RFM 병합 완료 → shape: (1143652, 6)
Recency 병합 완료 → shape: (1143652, 7)
활동성 90일 병합 완료 → shape: (1143652, 8)
활동성 180일 병합 완료 → shape: (1143652, 9)
구매 간격 병합 완료 → shape: (1143652, 11)
다양성 병합 완료 → shape: (1143652, 13)
취향 병합 완료 → shape: (1143652, 15)
[NEW] 구매 트렌드 병합 완료 → shape: (1143652, 17)
[NEW] 다양성 변화 병합 완료 → shape: (1143652, 19)
[NEW] 구매 리듬 병합 완료 → shape: (1143652, 21)
고객 메타 병합 완료 → shape: (1143652, 26)

최종 데이터 크기: (1143652, 26)
컬럼 목록: ['customer_id', 'churn', 'frequency', 'price_mean', 'price_sum', 'price_max', 'recency', 'frequency_last_90d', 'frequency_last_180d', 'avg_purchase_interval', 'std_purchase_interval', 'unique_articles', 'unique_product_groups', 'index_group_name', 'product_group_name', 'freq_decay_rate', 'recent_price_ratio', 'unique_articles_last_90d', 'diversity_decay_rate', 'last_interval', 'interval_acceleration', 'FN', 'Active', 'club_member_status', 'fashion_news_frequency', 'age']


## 7. 데이터 무결성 / 누수 검증

### 누수 검증 원칙
- 누수 기준: **cut_off 이후 데이터가 피처에 포함되었는가**
- 상관관계가 높은 피처 ≠ 누수 (이탈 신호를 담은 좋은 피처일 뿐)
- AUC > 0.95 이상이면 누수 의심 → 개별 피처 상관관계 재확인 필요


In [17]:
# 1. 중복 고객 확인
n_unique = df_analysis['customer_id'].nunique()
n_total  = len(df_analysis)
assert n_unique == n_total, '중복 고객 존재!'
print(f'중복 없음 (총 {n_total:,}명)')

# 2. 누수 컬럼 확인
leak_cols = ['recency_for_label','threshold','first_buy','last_buy']
found = [c for c in leak_cols if c in df_analysis.columns]
print(f'누수 컬럼: {found if found else "없음"}')

# 3. 상식 체크: 유지 고객이 이탈 고객보다 구매 횟수가 많아야 함
print('\n이탈 여부별 평균 구매 횟수 (유지=0이 높아야 정상)')
print(df_analysis.groupby('churn')['frequency'].mean().round(2))

# 4. 신규 피처 상관관계 확인 (0.95 이상이면 누수 의심)
new_features = ['freq_decay_rate','recent_price_ratio',
                'unique_articles_last_90d','diversity_decay_rate',
                'last_interval','interval_acceleration']

print('\n[NEW 피처] churn과의 상관관계 (절대값 0.95 이상이면 누수 의심)')
for col in new_features:
    if col in df_analysis.columns:
        corr = df_analysis[col].corr(df_analysis['churn'])
        flag = '⚠️ 누수 의심' if abs(corr) > 0.95 else '✅ 정상'
        print(f'  {col:<30}: {corr:.4f}  {flag}')


중복 없음 (총 1,143,652명)
누수 컬럼: 없음

이탈 여부별 평균 구매 횟수 (유지=0이 높아야 정상)
churn
0    31.38
1     8.67
Name: frequency, dtype: float64

[NEW 피처] churn과의 상관관계 (절대값 0.95 이상이면 누수 의심)
  freq_decay_rate               : -0.0778  ✅ 정상
  recent_price_ratio            : 0.0072  ✅ 정상
  unique_articles_last_90d      : -0.3309  ✅ 정상
  diversity_decay_rate          : -0.1360  ✅ 정상
  last_interval                 : -0.0281  ✅ 정상
  interval_acceleration         : 0.0059  ✅ 정상


## 8. 데이터 저장

In [18]:
file_name = './data/hnm_churn_dataset_v2.csv'
df_analysis.to_csv(file_name, index=False)

print(f'저장 완료: {file_name}')
print(f'최종 데이터 크기: {df_analysis.shape}')
print(f'이탈률: {df_analysis["churn"].mean()*100:.2f}%')
print(f'컬럼 수: v1({16}개) → v2({df_analysis.shape[1]}개) (+{df_analysis.shape[1]-16}개 추가)')


저장 완료: ./data/hnm_churn_dataset_v2.csv
최종 데이터 크기: (1143652, 26)
이탈률: 50.02%
컬럼 수: v1(16개) → v2(26개) (+10개 추가)


## 9. 최종 데이터셋 정의서 (v2)

모든 피처는 cut_off 기준일(2020-03-03) 이전 데이터만 사용하여 생성.

| 구분 | 컬럼명 | 설명 |
| :--- | :--- | :--- |
| 타겟 | churn | 이탈 여부 (1=이탈, 0=유지) |
| 식별자 | customer_id | 고객 ID |
| RFM | frequency | 총 구매 횟수 |
| RFM | price_mean | 평균 구매 금액 |
| RFM | price_sum | 총 구매 금액 |
| RFM | price_max | 최고 구매 금액 |
| RFM | recency | cut_off 기준 마지막 구매 경과일 |
| 활동성 | frequency_last_90d | 최근 90일 구매 횟수 |
| 활동성 | frequency_last_180d | 최근 180일 구매 횟수 |
| 구매 주기 | avg_purchase_interval | 평균 구매 간격 (일) |
| 구매 주기 | std_purchase_interval | 구매 간격 표준편차 |
| 다양성 | unique_articles | 구매 고유 상품 수 |
| 다양성 | unique_product_groups | 구매 고유 카테고리 수 |
| 취향 | index_group_name | 가장 많이 산 인덱스 그룹 |
| 취향 | product_group_name | 가장 많이 산 카테고리 |
| 메타 | age / FN / Active / club_member_status / fashion_news_frequency | 고객 메타 정보 |
| **[NEW]** 트렌드 | **freq_decay_rate** | 최근 90일 구매 / 최근 180일 구매 (1이면 변화 없음, 0.5면 절반으로 감소) |
| **[NEW]** 트렌드 | **recent_price_ratio** | 최근 90일 평균 구매액 / 전체 평균 구매액 (1 미만이면 지출 감소) |
| **[NEW]** 다양성 변화 | **unique_articles_last_90d** | 최근 90일 구매 고유 상품 수 |
| **[NEW]** 다양성 변화 | **diversity_decay_rate** | 최근 90일 고유 상품 수 / 전체 고유 상품 수 (낮을수록 스펙트럼 축소) |
| **[NEW]** 리듬 | **last_interval** | 마지막 구매 간격 (일) |
| **[NEW]** 리듬 | **interval_acceleration** | 마지막 구매 간격 / 평균 구매 간격 (1 초과면 최근 뜸해진 것) |


In [19]:
df_v2 = pd.read_csv('./data/hnm_churn_dataset_v2.csv')

# 신규 피처 기초 통계
new_features = ['freq_decay_rate', 'recent_price_ratio',
                'unique_articles_last_90d', 'diversity_decay_rate',
                'last_interval', 'interval_acceleration']

print(f'전체 shape: {df_v2.shape}')
print(f'컬럼 수: v1(16개) → v2({df_v2.shape[1]}개)\n')

print('--- 신규 피처 결측치 비율 ---')
for col in new_features:
    null_pct = df_v2[col].isnull().mean() * 100
    print(f'{col:<30}: {null_pct:.1f}%')

print('\n--- 신규 피처 기초 통계 ---')
print(df_v2[new_features].describe().round(4))

전체 shape: (1143652, 26)
컬럼 수: v1(16개) → v2(26개)

--- 신규 피처 결측치 비율 ---
freq_decay_rate               : 40.7%
recent_price_ratio            : 59.9%
unique_articles_last_90d      : 0.0%
diversity_decay_rate          : 0.0%
last_interval                 : 9.8%
interval_acceleration         : 32.6%

--- 신규 피처 기초 통계 ---
       freq_decay_rate  recent_price_ratio  unique_articles_last_90d  \
count      678535.0000         459155.0000              1.143652e+06   
mean            0.4514              1.0242              2.327800e+00   
std             0.3994              0.3864              4.901900e+00   
min             0.0000              0.0037              0.000000e+00   
25%             0.0000              0.8177              0.000000e+00   
50%             0.4167              1.0000              0.000000e+00   
75%             0.9000              1.1586              3.000000e+00   
max             1.0000             10.2690              1.890000e+02   

       diversity_decay_rate  last_i